In [31]:
import pandas as pd
import re
import numpy as np

In [ ]:
df = pd.read_csv(r'C:patients_m.csv')

print(df.shape)
print(df.dtypes)

(10000, 8)
patient_id                     str
age                          int64
gender                         str
blood_type                     str
chronic_conditions             str
insurance_type                 str
registration_date              str
treatment_adherence_pct    float64
dtype: object


In [3]:
df.head(10)

,patient_id,age,gender,blood_type,chronic_conditions,insurance_type,registration_date,treatment_adherence_pct
0,PAT-00001,40,Male,A+,Diabetes,Medicaid,1/13/2021,74.0
1,PAT-00002,59,Male,A+,Hypertension,Medicare,10/30/2021,65.1
2,PAT-00003,74,Male,B+,Hypertension,Uninsured,11/29/2021,76.1
3,PAT-00004,59,Male,A+,COPD & Heart Disease,Private,3/23/2022,91.8
4,PAT-00005,50,Female,O+,Chronic Kidney Disease,Medicaid,2/19/2017,75.2
5,PAT-00006,-1,Female,O+,Obesity,Uninsured,7/13/2021,63.7
6,PAT-00007,13,Male,A+,Hypertension,Private,12/27/2021,89.1
7,PAT-00008,53,Male,O+,Hypertension,Private,2/21/2019,87.2
8,PAT-00009,87,Male,O+,Asthma,Private,5/17/2018,72.4
9,PAT-00010,86,Female,A+,Hypertension,Medicaid,5/18/2020,70.1


In [20]:
# Check chronic_conditions columns delimeters
df['chronic_conditions'].value_counts()

chronic_conditions
Hypertension                         1289
Diabetes                             1212
Asthma                                931
Heart Disease                         852
COPD                                  816
Obesity                               787
Chronic Kidney Disease                620
Diabetes, Hypertension                211
Asthma, Obesity                       143
Hypertension, Heart Disease           143
COPD, Heart Disease                   115
Diabetes, Obesity                     107
Diabetes; Hypertension                 68
Diabetes,Hypertension                  63
Hypertension,Heart Disease             60
Diabetes|Hypertension                  58
Diabetes & Hypertension                54
Hypertension/Heart Disease             53
Diabetes/Hypertension                  51
Diabetes , Hypertension                51
Diabetes | Hypertension                50
Hypertension | Heart Disease           48
Hypertension , Heart Disease           45
Hypertension & 

In [22]:
# Normalize column chronic_conditions

def normalize_condition(val):
    if not isinstance(val, str) or val.strip() == "":
        return val
    # Split on any delimeter
    parts = re.split(r'[,;|/&]+', val)
    # Remove whitespace/blanks, capitalize first letter
    parts = [p.strip().title() for p in parts if p.strip()]
    # Sort unique values
    parts = sorted(set(parts))

    return ", ".join(parts)

In [24]:
df['chronic_conditions'] = df['chronic_conditions'].apply(normalize_condition)

In [25]:
df['chronic_conditions'].value_counts()

chronic_conditions
Hypertension                       1289
Diabetes                           1212
Asthma                              931
Heart Disease                       852
Copd                                816
Obesity                             787
Chronic Kidney Disease              620
Diabetes, Hypertension              606
Heart Disease, Hypertension         470
Asthma, Obesity                     383
Copd, Heart Disease                 314
Diabetes, Obesity                   288
Diabetes, Hypertension, Obesity     107
Name: count, dtype: int64

In [34]:
# Check age column outliers
print(df[df['age'] <= 0].shape)
print(df[df['age'] > 100].shape)

(173, 8)
(123, 8)


In [35]:
# Flag outliers before replacing 

MIN_AGE = 1
MAX_AGE = 100
df['age_outlier'] = ~ df['age'].between(MIN_AGE, MAX_AGE)

In [38]:
df[df['age_outlier']].shape

(296, 9)

In [39]:
# Replace outliers with NaN
df['age'] = df['age'].where(df['age'].between(MIN_AGE, MAX_AGE), other=np.nan)

In [41]:
# Check age column outliers
print(df[df['age'] <= 0].shape)
print(df[df['age'] > 100].shape)
print(df['age'].isna().sum())

(0, 9)
(0, 9)
296


In [42]:
df.info()

<class 'pandas.DataFrame'>
Index: 9817 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   patient_id               9817 non-null   str    
 1   age                      9521 non-null   float64
 2   gender                   9817 non-null   str    
 3   blood_type               9817 non-null   str    
 4   chronic_conditions       8675 non-null   str    
 5   insurance_type           9817 non-null   str    
 6   registration_date        9817 non-null   str    
 7   treatment_adherence_pct  9817 non-null   float64
 8   age_outlier              9817 non-null   bool   
dtypes: bool(1), float64(2), str(6)
memory usage: 699.8 KB


In [46]:
# Convert string column to datetime
df['registration_date'] = pd.to_datetime(df['registration_date'])
df.dtypes

patient_id                            str
age                               float64
gender                                str
blood_type                            str
chronic_conditions                    str
insurance_type                        str
registration_date          datetime64[us]
treatment_adherence_pct           float64
age_outlier                          bool
dtype: object

In [48]:
df.to_csv("patients.csv", index=False)